# GCN 튜닝 — Colab
계획서 4.1 그리드(lr × dropout × weight_decay, 8조합)를 **검증 PR-AUC 평균**으로 비교하고, 그 위에 patience 추가 실험을 한다. 테스트 구간(35~49)은 선택에 쓰지 않는다 (규칙 2).

- 7) 그리드, 8) patience 추가 실험, 9) 최종 본 학습 순서로 실행.
- 튜닝 결과는 `results_tune/`, 최종 결과는 `results/`에 저장된다.
- 그리드 밖 실험(patience)은 보고서에서 **별도 표**로 기록한다.


In [ ]:
# 1) 환경 확인
!python --version
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 T4 GPU로 바꾸세요'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) 저장소 가져오기 (private이면 GitHub 토큰 입력, public이면 엔터)
import getpass, os
if not os.path.exists('2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git
%cd 2026-2-HAI-ProjectTeam8
!git pull -q || true
!ls models common

In [ ]:
# 3) 데이터 복사 (드라이브 -> data/elliptic). 바로가기 위치가 달라도 찾도록 후보 경로를 순서대로 시도
import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = [
    "/content/drive/MyDrive/Colab Notebooks/elliptic",   # 마린 방식
    "/content/drive/MyDrive/elliptic",                    # 내 드라이브 루트에 바로가기를 둔 경우
]
NEED = ["elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"]
src = next((p for p in CANDIDATES if all(os.path.exists(f"{p}/{f}") for f in NEED)), None)
if src is None:
    for p in CANDIDATES:
        print(p, "->", os.listdir(p) if os.path.isdir(p) else "없음")
    raise FileNotFoundError("elliptic CSV 3개를 못 찾음: 공유받은 폴더를 '드라이브에 바로가기 추가' 했는지 확인")
os.makedirs("data/elliptic", exist_ok=True)
for f in NEED:
    shutil.copy(f"{src}/{f}", f"data/elliptic/{f}")
print("복사 완료:", src)
!ls -la data/elliptic


In [ ]:
# 4) PyTorch Geometric 설치 (Colab에는 torch가 이미 있음)
!pip install -q torch-geometric
import torch_geometric; print('torch_geometric', torch_geometric.__version__)

In [ ]:
# 5) 데이터 검증 (모두 OK여야 함)
!python scripts/verify_data.py data/elliptic

In [ ]:
# 6) 튜닝 공통 함수 (이 셀을 먼저 실행)
import itertools, subprocess, re, os
import numpy as np, pandas as pd
pd.set_option("display.width", 200)

def run_gcn(out_dir, lr, dropout, wd, patience=30, max_epochs=300, seeds=(0, 1, 2, 3, 4)):
    """models.gcn을 CLI로 한 번 돌리고 '검증 행'만 읽어 요약을 돌려준다. 테스트 행은 쓰지 않는다."""
    cmd = ["python", "-m", "models.gcn", "--data-root", "data/elliptic",
           "--seeds", *map(str, seeds), "--lr", str(lr), "--dropout", str(dropout),
           "--weight-decay", str(wd), "--patience", str(patience),
           "--max-epochs", str(max_epochs), "--results-dir", out_dir]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError("학습 실패: " + " ".join(cmd))
    df = pd.read_csv(f"{out_dir}/metrics/gcn.csv")
    v = df[df.split == "val"].copy()                       # 검증 행 = Train만 학습한 모델 기준
    v["best_epoch"] = v.notes.str.extract(r"best_epoch=(\d+)").astype(int)
    return dict(val_pr_auc_mean=v.pr_auc.mean(), val_pr_auc_std=v.pr_auc.std(),
                best_epoch_min=v.best_epoch.min(), best_epoch_max=v.best_epoch.max(),
                best_epoch_mean=v.best_epoch.mean())


In [ ]:
# 7) 계획서 8조합 그리드 (lr x dropout x weight_decay), 시드 0~4
# 선택 기준 = 검증 PR-AUC 평균. 테스트 구간은 보지 않는다.
# patience 30은 학습을 일찍 끊어 순위가 잡음에 흔들렸으므로 100/500으로 다시 돌린다.
# (이전 patience 30 결과는 results_tune/grid/ 에 있으니, 폴더를 patience별로 분리해 덮어쓰지 않는다)
GRID_PATIENCE, GRID_MAX_EPOCHS = 100, 500
rows = []
for lr, do, wd in itertools.product([0.01, 0.005], [0.3, 0.5], [5e-4, 1e-3]):
    tag = f"lr{lr}_do{do}_wd{wd}"
    print("==", tag)
    rows.append(dict(tag=tag, lr=lr, dropout=do, wd=wd,
                     **run_gcn(f"results_tune/grid_p{GRID_PATIENCE}/{tag}", lr, do, wd,
                               patience=GRID_PATIENCE, max_epochs=GRID_MAX_EPOCHS)))

grid = pd.DataFrame(rows).sort_values("val_pr_auc_mean", ascending=False).reset_index(drop=True)
os.makedirs("results_tune", exist_ok=True)
grid.to_csv(f"results_tune/grid_summary_p{GRID_PATIENCE}.csv", index=False)
print(grid.round(4).to_string())
gap = grid.val_pr_auc_mean.iloc[0] - grid.val_pr_auc_mean.iloc[1]
print(f"\n1위-2위 차이 {gap:.4f} / 1위 표준편차 {grid.val_pr_auc_std.iloc[0]:.4f}  "
      "(차이가 표준편차보다 작으면 우열을 주장하지 말 것)")


In [ ]:
# 8) 추가 실험 (계획서 그리드 밖, 보고서에는 별도 표로): 그리드 1위 조합에서 patience를 늘려 본다
# 가설: patience 30이 학습을 일찍 끊어 시드 편차가 커진다. 선택 기준은 역시 검증 PR-AUC.
best = grid.iloc[0]
extra = []
for pat, max_ep in [(30, 300), (100, 500)]:
    print("== patience", pat)
    extra.append(dict(patience=pat, max_epochs=max_ep,
                      **run_gcn(f"results_tune/patience/p{pat}", best.lr, best.dropout, best.wd,
                                patience=pat, max_epochs=max_ep)))
extra = pd.DataFrame(extra)
extra.to_csv("results_tune/patience_summary.csv", index=False)
print(f"기준 조합: lr={best.lr}, dropout={best.dropout}, wd={best.wd}")
print(extra.round(4).to_string(index=False))


In [ ]:
# 9) 최종 설정으로 시드 0~4 본 학습 -> results/ 에 저장 (gcn.csv, pred, emb 갱신)
# 아래 값은 7)·8) 결과를 보고 직접 정한다. 그리드 1위와 patience 실험 결과를 반영할 것.
LR, DROPOUT, WD = float(best.lr), float(best.dropout), float(best.wd)   # 기본값: 그리드 1위
PATIENCE, MAX_EPOCHS = GRID_PATIENCE, GRID_MAX_EPOCHS                     # 7)과 같은 patience (기본 100, 500)
print(f"최종 설정: lr={LR} dropout={DROPOUT} wd={WD} patience={PATIENCE} max_epochs={MAX_EPOCHS}")
!python -m models.gcn --data-root data/elliptic --lr {LR} --dropout {DROPOUT} --weight-decay {WD} --patience {PATIENCE} --max-epochs {MAX_EPOCHS}


In [ ]:
# 10) XGBoost와 비교표
!python scripts/collect_results.py


In [ ]:
# 11) 시간 단계별 F1: GCN vs XGBoost (35~42 vs 43~49)
fs = {m: f"results/timestep_f1_{m}.csv" for m in ("xgb", "gcn")}
t = {m: pd.read_csv(p).groupby("time_step").f1.mean().round(3) for m, p in fs.items() if os.path.exists(p)}
df = pd.DataFrame(t)
print(df.to_string())
print("\n평균 F1  35~42:", df.loc[35:42].mean().round(3).to_dict(), "| 43~49:", df.loc[43:49].mean().round(3).to_dict())


In [ ]:
# 12) 구간별 PR-AUC (35~42 / 43~49 / 전체). results/pred 의 예측값과 라벨만 쓰므로 재학습 불필요
from sklearn.metrics import average_precision_score
from common.data import load_data

d = load_data("data/elliptic", standardize=True)
segs = {"35~42": (35, 42), "43~49": (43, 49), "전체 35~49": (35, 49)}
rows = []
for seed in range(5):
    p = np.load(f"results/pred/pred_gcn_seed{seed}.npy")
    for name, (a, b) in segs.items():
        m = d.test_mask & (d.time_step >= a) & (d.time_step <= b)
        rows.append(dict(seg=name, seed=seed, pr_auc=average_precision_score(d.y[m], p[m]),
                         base=d.y[m].mean()))                      # 무작위 기준선 = 그 구간 불법 비율
seg_df = pd.DataFrame(rows)
out = seg_df.groupby("seg").agg(pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"), base=("base", "first"))
print(out.round(3).loc[list(segs)])


In [ ]:
# 13) 드라이브 백업. 마린이 올린 기존 gcn_results(튜닝 전)는 덮어쓰지 않도록 별도 폴더에 저장
DST = "/content/drive/MyDrive/Colab Notebooks/elliptic/gcn_results_tuned"
!mkdir -p "{DST}"
!cp -r results/metrics results/pred results/emb results/timestep_f1_gcn.csv results/summary.csv "{DST}"
# 튜닝 폴더는 emb(약 2.5GB, 하이브리드에 안 쓰임)를 빼고 복사. rsync 대신 shutil이라 Colab에서 바로 동작
import shutil
shutil.copytree("results_tune", f"{DST}/results_tune", ignore=shutil.ignore_patterns("emb"), dirs_exist_ok=True)
!ls "{DST}" "{DST}/results_tune"
